## 6. Final Assessment & Independent Exercise: Multi-Sensor Alternative Spectral Index & Time-Series Export


🎯 **Your Turn 12 (Final Synthesis Challenge):**

> Create a new Jupyter Notebook in your GitHub repository and replicate what you  consider is necessary to reach waht is required as follows:
>
>  1. Choose one index distinct from NDVI (e.g., **EVI**, **SAVI**, or **NDWI**) for canopy, soil, or moisture evaluation.
> 2.Generate annual median composites for both **Sentinel-2** and **Landsat 8** over your area, applying sensor-specific cloud masking (SCL / QA_PIXEL) and Landsat 8 Surface Reflectance scaling factors.
> 3. **Construct Multi-Temporal Stacks:** Combine the 10 annual index layers (2016 to 2025) into a single 10-band `ee.Image` stack for Sentinel-2 and a corresponding 10-band stack for Landsat 8 (renaming bands sequentially as `Index_2016`, `Index_2017`, ..., `Index_2025`).
> 4. **GeoTIFF Export:** Export both 10-band image stacks to Google Drive as projected GeoTIFF rasters using the official local CRS for Colombia (**EPSG:9377**).

> 5. Commit and push your final `.ipynb` file to your public GitHub repository.
> 6. Ensure your notebook contains executed output cells, structured Markdown headers, and code comments explaining your custom study area selection.
> 7. Copy the direct link to your Jupyter Notebook on GitHub and submit it using the following link:
>
> 📋 [**Submit Your Final Notebook Link Here**](https://forms.gle/73ypriddjvWtarV8A)

---

In [ ]:
import geopandas as gpd # Mnipular dataframes
import rasterio # Manipular raster datasets como arreglos numpy
import rasterio.mask # Modulo de rasterio para cortar raster con formas vector
import rasterio.warp # Modulo de rasterio para automatizar funciones de resampleo 
from rasterio.enums import Resampling # Algoritmos nativos de rasterio de remuestreo o interpolación para resampleo 
import ee # API de google earth engine
import numpy as np # Manipular arreglos
import matplotlib.pyplot as plt # Manipular y crear gráficos
from pathlib import Path # Gestion de rutas de archivos
import geemap # herramientas para mostrar en mapa las imágenes generadas en GEE

In [ ]:
ee.Authenticate()

In [ ]:
try:
    ee.Initialize()
    print("Google Earth Engine initialised successfully.")
except Exception as e:
    print(f"Error initialising GEE: {e}")

In [ ]:
root_folder=Path(r"/notebooks/GEOPROCESAMIENTO_UNAL")
# Load Colombian municipalities
gdf = gpd.read_file(root_folder /"municipios_colombia.gpkg")
df_muni = gdf[gdf["MPIO_CNMBR"] == "LA MACARENA"].copy()

In [ ]:
bbox = gdf_muni.to_crs(epsg=4326).total_bounds
ee_bounds = ee.Geometry.BBox(bbox[0], bbox[1], bbox[2], bbox[3])


geojson_geom = gdf_muni.to_crs(epsg=4326).geometry.iloc[0].__geo_interface__
ee_muni_geom = ee.Geometry(geojson_geom)

In [ ]:
def mask_s2_clouds_scl(image):
    """Masks clouds, cloud shadows, and cirrus using the SCL band."""
    scl = image.select('SCL')
    
    # Identify unwanted pixel classes
    cloud_shadows = scl.eq(3)
    clouds_medium = scl.eq(8)
    clouds_high = scl.eq(9)
    cirrus = scl.eq(10)
    
    # Combine all mask conditions (1 = invalid pixel)
    mask = cloud_shadows.Or(clouds_medium).Or(clouds_high).Or(cirrus).Not()
    
    # Update image mask and retain properties
    return image.updateMask(mask)

In [ ]:

# 1. Rango de años para las imágenes sentinel 2 (2016 to 2025)
years = ee.List.sequence(2016, 2025)

# 2. Funcion para calcular el NDWI para cada año del rango
def compute_annual_ndwi(year):
    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year, 12, 31)
    
    annual_s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_muni_geom)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
        .map(mask_s2_clouds_scl)
        .median()
    )
    nombre_banda = ee.String("NDWI_").cat(ee.Number(year))
    
    annual_ndwi = (
        annual_s2
        .normalizedDifference(["B3", "B8"])
        .set("year", year)
    )
    
    return annual_ndwi.rename([nombre_banda])

# Construye una colección de los NDWI de cada año
annual_ndwi_collection = ee.ImageCollection(years.map(compute_annual_ndwi))

s2_nombres_bandas = years.map(lambda y: ee.String("NDWI_").cat(ee.Number(y)))

s2_stack = annual_ndwi_collection.toBands().rename(s2_nombres_bandas)

In [ ]:
# Exportar la colección de imágenes a Google Drive

task_s2 = ee.batch.Export.image.toDrive(
    image=s2_stack,
    description='Sentinel2_NDWI_Stack_2016_2025',
    folder='GEOPROCESAMIENTO_UNAL', 
    region=ee_muni_geom,
    scale=10, 
    crs='EPSG:9377',
    maxPixels=1e13
)
task_s2.start()

print("Exportado Stack Sentinel 2")

In [ ]:
def mask_landsat8_clouds(image):
 # Mascara de nubes y sombras usando la banda QA_PIXEL de Landsat 8
    qa = image.select('QA_PIXEL')
    # Bits 3 y 4 representan nubes y sombras de nubes respectivamente
    cloud_shadow = qa.bitwiseAnd(1 << 4).eq(0)
    clouds = qa.bitwiseAnd(1 << 3).eq(0)
    return image.updateMask(cloud_shadow.And(clouds))

In [ ]:
# Escalado para pasar de valores digitales a valores de reflectancia para Landsat 8
def scale_landsat8(image):
    optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    return image.addBands(optical_bands, None, True)

years = ee.List.sequence(2016, 2025)

# 2. Funcion para calcular el NDWI para cada año del rango
def compute_annual_ndwi_land8(year):
    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year, 12, 31)
    
    landsat8_collection = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(ee_muni_geom)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.lt("CLOUD_COVER", 30))
        .map(mask_landsat8_clouds) 
        .map(scale_landsat8)
)

    nombre_banda_landsat8 = ee.String("NDWI_").cat(ee.Number(year))
    
    annual_ndwi_land8 = ( landsat8_collection
        .median()
        .normalizedDifference(['SR_B3', 'SR_B5'])
        .set("year", year)
    )
    
    return annual_ndwi_land8.rename([nombre_banda_landsat8])

# Construye una colección de los NDWI para Landsat de cada año
annual_ndwi_collection = ee.ImageCollection(years.map(compute_annual_ndwi_land8))
landsat8_nombres_bandas = years.map(lambda y: ee.String("NDWI_").cat(ee.Number(y)))
l8_stack = annual_ndwi_collection.toBands().rename(landsat8_nombres_bandas)

In [ ]:
task_l8 = ee.batch.Export.image.toDrive(
    image=l8_stack,
    description='Landsat8_NDWI_Stack_2016_2025',
    folder='GEOPROCESAMIENTO_UNAL',
    region=ee_muni_geom,
    scale=30,
    crs='EPSG:9377',
    maxPixels=1e13
)
task_l8.start()
print("Exportado Stack Landsat 8")